In [5]:
from imports import *
from quick_class import *
import warnings
import fnmatch
from obspy.core.inventory.inventory import read_inventory
import operator
from obspy import read
# ========================================================================================================================================================
from IPython.display import clear_output

instrument_colors = {'B2':[227,26,28], 'KE':[178,223,138], 'AB':[166,206,227], 'BA':[202,178,214], 'AR':[255,127,0], 'TRM':[31,120,180], 'BG':[51,160,44], 'BD':[106,61,154]}
_ = [instrument_colors.update({k:list(np.array(instrument_colors[k])/255)}) for k in list(instrument_colors.keys())]
seismometer_marker = {'Guralp CMG3T 120':'o','Trillium 240':'x','Trillium Compact':'^'}

def write_pickle(file,var):
    import pickle
    with open(str(file), 'wb') as handle:
        pickle.dump(var, handle, protocol=pickle.HIGHEST_PROTOCOL)
    print('Saved to :' + str(file))
def load_pickle(file):
    import pickle
    with open(file, 'rb') as handle:
        b = pickle.load(handle)
    return b
def mirror_events(reports):
    nkeys = [n for n in list(reports[0].__dict__.keys()) if not n=='f']
    mirror = dict()
    for ni,n in enumerate(nkeys):
        skeys = list(reports[0][n].__dict__.keys())
        for si,s in enumerate(skeys):
            stanm = '.'.join([n,s]).replace('n','')
            ev0 = [k.replace('.','') for k in reports[0][n][s].events]
            ev1 = [k.replace('.','') for k in reports[1][n][s].events]
            mirror[stanm] = np.intersect1d(ev0,ev1)
    return mirror
# ================================================================================================================================
from matplotlib.collections import PatchCollection
from matplotlib.patches import Rectangle
reportfolder = Path('/Users/charlesh/Documents/Codes/OBS_Methods/NOISE/ATACR_HPS_Comp/_DataArchive/Analysis/NetworkCoherences')
bands = ['1-10','10-30','30-100']
# ================================================================================================================================
# ================================================================CODE SNIPPETS===========================================================================
# for a in cm._cmap_names_categorical:
#     display(cm.__dict__[a].resampled(4))
# for (Event,Station,Metrics,Comp) in OBS_Generator(catalog,dirs['Py_DataParentFolder']):
#     print(Event)
# for i,(Event,Station,Metrics,Comp) in zip(range(1),OBS_Generator(catalog,dirs['Py_DataParentFolder'])):
#     print(Event)
# # ======================================================================================================================================================
def smooth(d,k=10):return np.convolve(d, np.ones(k) / k, mode='same')
NoiseColors = [mcolors.to_hex(m) for m in [cm.__dict__[e].resampled(30).resampled(6).colors for e in ['devon_categorical']][0]]
# [display(c) for c in [cm.__dict__[e].resampled(70).resampled(5) for e in ['nuuk_categorical','devon_categorical','hawaii_categorical','imola_categorical','lapaz_categorical']]]
# np.array([[mcolors.to_hex(c) for c in r] for r in [cm.__dict__[e].resampled(70).resampled(5).colors for e in ['nuuk_categorical','devon_categorical','hawaii_categorical','imola_categorical','lapaz_categorical']]])
# display(catalog)
print('Stations: ' + str(len(catalog)))
# catalog = catalog.iloc[np.where(catalog.Station=='M08A')[0][0]].to_frame().T
# _ = [print('['+(str(np.round(100*100*i/len(catalog))/100)).zfill(5) + '] ' + n+ '.' +s) for i,s,n in zip(range(len(catalog)),catalog.Station,catalog.Network)]
# _ = [print(s) for s in ('[' + catalog.Network + '] | ' + catalog.Experiment).unique()]

Stations: 96


In [11]:
(catalog.Experiment + ' | ' + catalog.Network).unique()

array(['ALBACORE | 2D', 'CASCADIA KECK | 7A', 'CASCADIA INITIATIVE | 7D',
       'BLANCO | X9', 'MARIANA | XF', 'AACSE | XO', 'LAU | YL',
       'ENAM | YO', 'PLATE | Z6', 'NOMELT | ZA', 'PAPUA | ZN'],
      dtype=object)

In [6]:
# bng_average --plot --save YH_list.pkl

In [7]:
# [display(cm.cmaps[e].resampled(8).resampled(4)) for e in list(['devon_categorical','hawaii_categorical'])]
# colors = np.array([cm.cmaps[e].resampled(8).resampled(4).colors for e in list(['devon_categorical','hawaii_categorical'])]).reshape((8,4))
# colors_hex = [mcolors.to_hex(c) for c in colors]

In [8]:
# from obspy import read
# f = '/Users/charlesh/Documents/Codes/OBS_Methods/NOISE/ATACR_HPS_Comp/_DataArchive/DLOPY_Data/EVENTS/2D.OBS19/2011.065.12.31.HZ.SAC'
# st = read(f)
# print(st[0].stats.endtime - st[0].stats.starttime)
# print(st[0].stats.sampling_rate)

In [ ]:
# dldata = DL(sta)
# # Add event to object
# accept = dldata.add_event(ev, gacmin=args.mindist, gacmax=args.maxdist,depmax=args.maxdep, returned=True)
# has_data = dldata.download_data(client=wf_client, stdata=args.localdata,ndval=args.ndval, new_sr=2., t1=t1, t2=t2,returned=True, verbose=args.verb)
# dldata.calc(showplot=False)
ev_intputfolder = Path(dirs['Py_CorrectedTraces'])
stakey = '7D.M08A'


cat = catalog[catalog.StaName==stakey]
staquery_output = build_staquery(cat,staquery_output=None)
# Load catalog
events = [ev[0] if isinstance(ev,obspy.core.event.catalog.Catalog) else ev for ev in cat.iloc[0].Metadata]
evcat = Catalog();evcat.extend(events)
evtimes = [e.origins[0].time for e in evcat]

# Load data
dateformat = '%Y.%j.%H.%M'
ev = evtimes[0]
ev.strftime(dateformat)
st,inv = Stream(),[]
files = [ev.strftime(dateformat)+'.*'+c+'.SAC' for c in ['Z','1','2']]
[(st.append(tr[0]),inv.append(invi)) for tr,invi in [load_sac(ev_intputfolder / stakey / f) for f in files]];inv = inv[0]
# Plot event list
evplot = evcat.plot()
fig = inv.plot(show=False,color='r',block=True,draw=False)
plt.show()
clear_output(wait=False)
evcat.plot(fig=fig)

dldata = DL(staquery_output[staquery_output.keys()[0]])
# mindist = 5
# maxdist = 175
# maxdep = 1000
dldata.add_event(events[0], returned=True)

In [ ]:
attr_sta = AttribDict()
# staquery_output.to_dict()
attr_sta.update

In [ ]:
staquery_output[staquery_output.keys()[0]]

In [ ]:
def plot_spec_coh_adm_ph(Metrics):
    pairs = ['ZP']
    meters = ['psd','Coherence','Admittance','Phase']
    fig, axes = plt.subplots(nrows=4, ncols=1,figsize=(8,10),layout='constrained',squeeze=False,sharey='row',sharex='all')
    axes = axes.reshape(-1)
    stam = Metrics['ATaCR'].traces[0].stats.network + '.' + Metrics['ATaCR'].traces[0].stats.station
    label = Metrics['ATaCR'].traces.select(channel='*Z')[0].stats.location
    Pre = Metrics['Raw']
    Post = Metrics['ATaCR']
    Noise = Metrics['Noise']
    fn = fnotch(abs(Post.traces[0].stats.sac.stel*1000))
    tstamp = Pre.traces[0].stats.starttime.strftime('%Y.%j.%H.%M')
    for pi,(ax,m) in enumerate(zip(axes,meters)):
        if m=='psd':
            p = 'Z'
        else:
            p = pairs[0]
        evf,prey = Pre.__getattribute__(m)(p)
        evf,posty = Post.__getattribute__(m)(p)
        if m=='psd':
            noisef,noisey = Noise.f,Noise.StaNoise.power.__dict__['cZZ']
        else:
            noisef,noisey = Noise.__getattribute__(m)(p)
        noisey = noisey[noisef>0]
        noisef = noisef[noisef>0]
        if m=='psd':
            noisey = 10*np.log10(noisey)
            prey = 10*np.log10(prey)
            posty = 10*np.log10(posty)
        ax.scatter(noisef,noisey,s=0.5,c='gray',label='Noise')
        if pi==0:
            lbl = stam + ' | ' + label + ' ' + tstamp + ' | '
        else:
            lbl = ''
        ax.set_xlabel('Frequency')
        ax.set_ylabel(m.replace('psd','Power Density'))
        ax.set_title(lbl + p + '-' + m.replace('psd','PSD'),fontweight='bold')
        ax.scatter(evf,prey,c='k',label='PRE',marker='o',s=1)
        ax.scatter(evf,posty,c='m',label='POST',marker='o',s=0.5)
        ax.axvline(fn,linewidth=0.2,color='k')
        if pi==0:
            ax.text(fn*1.05,0.99*min(ax.get_ylim()),'Fn:' + str(round(1/fn*100)/100) + 's',alpha=0.4)
            ax.set_xscale('log')
            ax.set_xlim(evf[1],evf[-1])
            ax.legend(markerscale=10,ncols=len(meters))
    plt.tight_layout()
    return fig

In [ ]:
g

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ++++++++++++++++++++++++++ CONSTRUCTOR AREA ++++++++++++++++++++++++++++++
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
methods = ['PostATACR']
atacrdatafolder = archive / 'ATaCR_Data' / 'ATaCR_Python'
for correction_method in methods:
    coh_comp = correction_method.replace('PostHPS','HPS').replace('PostATACR','ATaCR')
    if correction_method=='PostHPS':
      return_hps = True
    else:
      return_hps = False
    OutFolder = Path(plotfolder)
    SubFolders = Path('EventRecords') / correction_method / 'coherence'
    OutFolder = OutFolder / SubFolders
    OutFolder.mkdir(parents=True,exist_ok=True)
    for station in catalog.iloc:
      stations = [station.Station]
      networks = [station.Network]
      events = station.Events
      for i,(net,sta) in enumerate(zip(networks,stations)):
        Metrics = []
        for evi,event in enumerate(events):
          depth = round(station.Metadata[evi].origins[0].depth/1000)
          mag = station.Metadata[evi].magnitudes[0].mag
          File = '.'.join([net,sta]) + '.m' + str(mag) + '.z' + str(depth) + 'km' + '.' + event + '.' + correction_method.replace('Post','') + '_SPECCOHPHADM.png'
          title = File.replace('_',' | ').replace('z','z: ').replace('m','mag: m')
          print('[' + str(evi) + '/' + str(len(events)) + '] ' + File)
          post_record = Stream()
          pre_record = Stream()
          M,Comp = get_metrics_comp(net,sta,atacrdatafolder,event,return_hps=return_hps,events_folder='EVENTS')
          # M['Noise'] = get_Noise(atacrdatafolder,net,sta,'sta')['Noise']
          Metrics.append(M.copy())
          fig = plot_spec_coh_adm_ph(M)
          save_tight(str(plotfolder / 'MeetingFigs' / 'SPECCOHPHADM' / File),fig,dpi=600)


In [ ]:
bplot['boxes'][0]._edgecolor